# Tarea ETL — Proceso de Movimientos de inventario (WWImporters)
**Entregable 2 — Implementación del ETL en PySpark**

Este notebook implementa el proceso ETL que puebla en la bodega de datos las dimensiones **Proveedor**, **TipoTransaccion** y **Fecha**, y la **tabla de hechos Movimiento**, a partir de la base transaccional `WWImportersTransactional`.

Las dimensiones **Producto** y **Cliente** NO se procesan aquí: son **dimensiones compartidas (conformadas)** que ya existen en la bodega gracias al proceso de Órdenes; el hecho solo hace *lookup* sobre ellas para traer sus llaves surrogadas.

Cada bloque sigue el flujo **Extraer → Transformar → Cargar** y aplica las reglas de negocio entregadas por WWImporters.

## 0. Configuración y utilidades

In [ ]:
# Configuración de conexión
# Use su usuario "Estudiante_i" y la contraseña asignada en el Excel de conexión del curso.
db_user = ''
db_psswd = ''

# Base transaccional (origen) y bodega de datos (destino).
# AJUSTE el nombre de su bodega si es distinto.
BODEGA = 'DB_202613_f_cucina'

source_db_connection_string = 'jdbc:mysql://157.253.236.116:8080/WWImportersTransactional'
dest_db_connection_string   = 'jdbc:mysql://157.253.236.116:8080/' + BODEGA

# Driver de conexión (ajuste la ruta si es necesario)
path_jar_driver = 'C:\\Program Files (x86)\\MySQL\\Connector J 8.0\\mysql-connector-java-8.0.28.jar' 

In [ ]:
import os
from pyspark.sql import functions as f, SparkSession, types as t
from pyspark import SparkContext, SparkConf, SQLContext
from pyspark.sql.functions import (col, when, regexp_replace, coalesce, to_timestamp,
                                   to_date, date_format, year, month, dayofmonth, weekofyear)

In [ ]:
# Configuración de la sesión Spark
conf = SparkConf().set('spark.driver.extraClassPath', path_jar_driver)
spark_context = SparkContext(conf=conf)
sql_context = SQLContext(spark_context)
spark = sql_context.sparkSession

In [ ]:
# Funciones de conexión: leer de la fuente y guardar en la bodega
def obtener_dataframe_de_bd(db_connection_string, sql, db_user, db_psswd):
    return spark.read.format('jdbc') \
        .option('url', db_connection_string) \
        .option('dbtable', sql) \
        .option('user', db_user) \
        .option('password', db_psswd) \
        .option('driver', 'com.mysql.cj.jdbc.Driver') \
        .load()

def guardar_db(db_connection_string, df, tabla, db_user, db_psswd):
    df.select('*').write.format('jdbc') \
        .mode('append') \
        .option('url', db_connection_string) \
        .option('dbtable', tabla) \
        .option('user', db_user) \
        .option('password', db_psswd) \
        .option('driver', 'com.mysql.cj.jdbc.Driver') \
        .save()

## BLOQUE 1 — Dimensión Proveedor
Fuente: `proveedores` + `CategoriasProveedores`.

### Extracción

In [ ]:
sql_proveedores = """(SELECT DISTINCT ProveedorID, NombreProveedor, CategoriaProveedorID,
       PersonaContactoPrincipalID, DiasPago, CodigoPostal
       FROM WWImportersTransactional.proveedores) AS Temp_proveedores"""

sql_categorias = """(SELECT DISTINCT CategoriaProveedorID, CategoriaProveedor
       FROM WWImportersTransactional.CategoriasProveedores) AS Temp_categorias"""

proveedores = obtener_dataframe_de_bd(source_db_connection_string, sql_proveedores, db_user, db_psswd)
categorias  = obtener_dataframe_de_bd(source_db_connection_string, sql_categorias,  db_user, db_psswd)
proveedores.show(5)

### Transformación
- **Regla:** unir con `CategoriasProveedores` para traer la categoría.
- **Regla:** los días de pago no pueden ser negativos → multiplicar por -1.
- **Regla:** proveedores con el mismo nombre + sufijo "Inc"/"Ltd" se unifican en uno solo (error de digitación).
- **Regla:** el código postal igual para todos ya fue corregido en la fuente → se toma tal cual.
- Se genera la llave surrogada `ID_Proveedor_DWH`.

In [ ]:
# 1. Unir con categorías (left join por CategoriaProveedorID)
proveedores = proveedores.join(categorias, how='left', on='CategoriaProveedorID')

# 2. Corregir días de pago negativos (x -1)
proveedores = proveedores.withColumn(
    'DiasPago', when(col('DiasPago') < 0, col('DiasPago') * -1).otherwise(col('DiasPago')))

# 3. Unificar proveedores con sufijo "Inc"/"Ltd": se limpia el nombre y se eliminan duplicados
proveedores = proveedores.withColumn(
    'NombreProveedor', f.trim(regexp_replace(col('NombreProveedor'), r'\s+(Inc|Ltd)\.?$', '')))
proveedores = proveedores.dropDuplicates(['NombreProveedor'])

# 4. (Código postal ya viene corregido desde la fuente: no requiere transformación)

# 5. Renombrar al esquema de la bodega y generar la llave surrogada
proveedores = proveedores.selectExpr(
    'ProveedorID as ID_Proveedor_T',
    'NombreProveedor as Nombre',
    'CategoriaProveedor as Categoria',
    'PersonaContactoPrincipalID as Contacto_principal',
    'DiasPago as Dias_pago',
    'CodigoPostal as Codigo_postal')
proveedores = proveedores.coalesce(1).withColumn(
    'ID_Proveedor_DWH', (f.monotonically_increasing_id() + 1).cast('int'))
proveedores = proveedores.select('ID_Proveedor_DWH', 'ID_Proveedor_T', 'Nombre',
                                 'Categoria', 'Contacto_principal', 'Dias_pago', 'Codigo_postal')
proveedores.show(5)

Registro comodín `ID = 0` (representa "sin dato"; lo usa el hecho cuando no hay coincidencia).

In [ ]:
prov_0 = spark.createDataFrame([(0, 0, 'Sin dato', 'Sin dato', 0, 0, 0)], schema=proveedores.schema)
proveedores = proveedores.unionByName(prov_0).orderBy(col('ID_Proveedor_DWH'))
proveedores.show(5)

### Carga
**OJO:** antes de guardar, asegúrese de que la tabla `Proveedor` esté vacía para no duplicar datos.

In [ ]:
guardar_db(dest_db_connection_string, proveedores, BODEGA + '.Proveedor', db_user, db_psswd)

## BLOQUE 2 — Dimensión TipoTransaccion
Fuente: `TiposTransaccion` (ya analizada por el grupo de consultores; se usa tal cual).

### Extracción

In [ ]:
sql_tipos = """(SELECT DISTINCT TipoTransaccionID, TipoTransaccionNombre
       FROM WWImportersTransactional.TiposTransaccion) AS Temp_tipos"""
tipos = obtener_dataframe_de_bd(source_db_connection_string, sql_tipos, db_user, db_psswd)
tipos.show()

### Transformación
Renombrar al esquema de la bodega y generar la llave surrogada.

In [ ]:
tipos = tipos.selectExpr('TipoTransaccionID as ID_Tipo_transaccion_T',
                         'TipoTransaccionNombre as Tipo')
tipos = tipos.coalesce(1).withColumn(
    'ID_Tipo_transaccion_DWH', (f.monotonically_increasing_id() + 1).cast('int'))
tipos = tipos.select('ID_Tipo_transaccion_DWH', 'ID_Tipo_transaccion_T', 'Tipo')

# Registro comodín ID = 0
tipos_0 = spark.createDataFrame([(0, 0, 'Sin dato')], schema=tipos.schema)
tipos = tipos.unionByName(tipos_0).orderBy(col('ID_Tipo_transaccion_DWH'))
tipos.show()

### Carga

In [ ]:
guardar_db(dest_db_connection_string, tipos, BODEGA + '.TipoTransaccion', db_user, db_psswd)

## BLOQUE 3 — Dimensión Fecha
Fuente: la columna `FechaTransaccion` de `movimientos` (viene como **texto**).

### Extracción

In [ ]:
sql_fechas = """(SELECT DISTINCT FechaTransaccion
       FROM WWImportersTransactional.movimientos) AS Temp_fechas"""
fechas = obtener_dataframe_de_bd(source_db_connection_string, sql_fechas, db_user, db_psswd)

# Revise el formato real del texto para ajustar el parseo si hace falta
fechas.select('FechaTransaccion').show(10, truncate=False)

### Transformación
- **Regla:** estandarizar el formato (YYYY-MM-DD HH:MM:SS si hay hora; si no, YYYY-MM-DD).
- **Regla:** los años anteriores a 2014 ya están incluidos (no se filtra por año).
- Derivar `Dia`, `Mes`, `Anio`, `Numero_semana_ISO` y generar `ID_Fecha` (entero YYYYMMDD).

In [ ]:
# Parseo robusto: intenta varios formatos de texto y se queda con el que funcione
fechas = fechas.withColumn('FechaStd', coalesce(
    to_timestamp(col('FechaTransaccion'), 'yyyy-MM-dd HH:mm:ss'),
    to_timestamp(col('FechaTransaccion'), 'yyyy-MM-dd'),
    to_timestamp(col('FechaTransaccion'), 'MMM dd,yyyy'),
    to_timestamp(col('FechaTransaccion'), 'M/d/yyyy')))

fechas = fechas.withColumn('Fecha', to_date(col('FechaStd')))
fechas = fechas.dropna(subset=['Fecha']).dropDuplicates(['Fecha'])

fechas = fechas \
    .withColumn('ID_Fecha', date_format(col('Fecha'), 'yyyyMMdd').cast('int')) \
    .withColumn('Dia', dayofmonth(col('Fecha'))) \
    .withColumn('Mes', month(col('Fecha'))) \
    .withColumn('Anio', year(col('Fecha'))) \
    .withColumn('Numero_semana_ISO', weekofyear(col('Fecha')))

fechas = fechas.select('ID_Fecha', 'Fecha', 'Dia', 'Mes', 'Anio', 'Numero_semana_ISO')

# Registro comodín ID = 0 (fecha desconocida)
from datetime import date
fecha_0 = spark.createDataFrame([(0, date(1900, 1, 1), 0, 0, 0, 0)], schema=fechas.schema)
fechas = fechas.unionByName(fecha_0).orderBy(col('ID_Fecha'))
fechas.show(5)

### Carga

In [ ]:
guardar_db(dest_db_connection_string, fechas, BODEGA + '.Fecha', db_user, db_psswd)

## BLOQUE 4 — Tabla de hechos Movimiento
Fuente: `movimientos` + las dimensiones ya cargadas (lookup para traer las llaves surrogadas `_DWH`).

### Extracción

In [ ]:
sql_movimientos = """(SELECT DISTINCT ProductoID, ClienteID, ProveedorID, TipoTransaccionID,
       FechaTransaccion, Cantidad
       FROM WWImportersTransactional.movimientos) AS Temp_movimientos"""
mov = obtener_dataframe_de_bd(source_db_connection_string, sql_movimientos, db_user, db_psswd)
print((mov.count(), mov.distinct().count()))
mov.show(5)

### Transformación
- **Regla:** eliminar las filas totalmente duplicadas (`dropDuplicates`).
- **Regla:** la regla de "máx. 50 millones por transacción" fue eliminada → no se filtra por cantidad.
- **Regla:** las cantidades negativas representan salidas de inventario → se conservan.
- Derivar `ID_Fecha` (YYYYMMDD) a partir de `FechaTransaccion`.
- *Lookup* (left join) a cada dimensión por su llave de negocio (`_T`) para traer las llaves surrogadas (`_DWH`); usar el comodín `0` cuando no haya coincidencia.

In [ ]:
# 1. Eliminar duplicados totales
mov = mov.dropDuplicates()

# 2. (No se filtra por cantidad: la regla de 50M fue eliminada)
# 3. (Se conservan las cantidades negativas: son salidas de inventario)

# 4. Derivar ID_Fecha (mismo parseo que en Dim_Fecha)
mov = mov.withColumn('FechaStd', coalesce(
    to_timestamp(col('FechaTransaccion'), 'yyyy-MM-dd HH:mm:ss'),
    to_timestamp(col('FechaTransaccion'), 'yyyy-MM-dd'),
    to_timestamp(col('FechaTransaccion'), 'MMM dd,yyyy'),
    to_timestamp(col('FechaTransaccion'), 'M/d/yyyy')))
mov = mov.withColumn('ID_Fecha', date_format(to_date(col('FechaStd')), 'yyyyMMdd').cast('int'))
mov = mov.fillna({'ID_Fecha': 0})

In [ ]:
# 5. Leer las dimensiones desde la bodega para el lookup de llaves surrogadas.
#    Producto y Cliente son dimensiones conformadas que YA existen en la bodega.
dim_producto = obtener_dataframe_de_bd(dest_db_connection_string,
    '(SELECT ID_Producto_DWH, ID_Producto_T FROM ' + BODEGA + '.Producto) AS dp', db_user, db_psswd)
dim_cliente = obtener_dataframe_de_bd(dest_db_connection_string,
    '(SELECT ID_Cliente_DWH, ID_Cliente_T FROM ' + BODEGA + '.Cliente) AS dc', db_user, db_psswd)
dim_proveedor = obtener_dataframe_de_bd(dest_db_connection_string,
    '(SELECT ID_Proveedor_DWH, ID_Proveedor_T FROM ' + BODEGA + '.Proveedor) AS dpr', db_user, db_psswd)
dim_tipo = obtener_dataframe_de_bd(dest_db_connection_string,
    '(SELECT ID_Tipo_transaccion_DWH, ID_Tipo_transaccion_T FROM ' + BODEGA + '.TipoTransaccion) AS dt', db_user, db_psswd)

In [ ]:
# Lookups (left join). La llave de negocio del origen se cruza con la _T de cada dimensión.
hecho = mov \
    .join(dim_producto,  mov.ProductoID == dim_producto.ID_Producto_T, 'left') \
    .join(dim_cliente,   mov.ClienteID == dim_cliente.ID_Cliente_T, 'left') \
    .join(dim_proveedor, mov.ProveedorID == dim_proveedor.ID_Proveedor_T, 'left') \
    .join(dim_tipo,      mov.TipoTransaccionID == dim_tipo.ID_Tipo_transaccion_T, 'left')

# Comodín 0 cuando no hubo coincidencia en la dimensión
hecho = hecho.fillna({
    'ID_Producto_DWH': 0, 'ID_Cliente_DWH': 0,
    'ID_Proveedor_DWH': 0, 'ID_Tipo_transaccion_DWH': 0})

# 6. Seleccionar las llaves foráneas + la medida Cantidad
hecho = hecho.select('ID_Fecha', 'ID_Producto_DWH', 'ID_Proveedor_DWH',
                     'ID_Cliente_DWH', 'ID_Tipo_transaccion_DWH', 'Cantidad')
hecho.show(5)

### Carga

In [ ]:
guardar_db(dest_db_connection_string, hecho, BODEGA + '.Hecho_Movimiento', db_user, db_psswd)

## Verificación
Verifique en MySQL Workbench que las cuatro tablas (`Proveedor`, `TipoTransaccion`, `Fecha`, `Hecho_Movimiento`) quedaron creadas y pobladas en la bodega `DB_202613_f_cucina`. Recuerde mantener las tablas para que los tutores puedan validarlas.